### Install project dependencies

**Purpose:** Install the packages required for local and Databricks demonstrations.

**Inputs:** `setup/requirements.txt`.

**Outputs:** A notebook environment with the declared project dependencies.

**Why it matters:** Reproducible dependencies keep setup and validation behavior aligned with the bundle Job.

In [0]:
# Purpose: install the Python packages required by Demo2_Olist.

%pip install -r /Workspace/Users/parvinbadalov@softserve.academy/Databricks-Academy-Lakehouse/Demos/Demo2_Olist/setup/requirements.txt

### Install the project requirements

**Purpose:** Install the declared Demo2 Olist Python dependencies.

**Inputs:** The project requirements file.

**Outputs:** Packages available to the notebook session.

**Why it matters:** Dependency setup makes local demonstrations reproducible.

In [0]:
# Purpose: restart Python so the installed packages become available.

dbutils.library.restartPython()

### Confirm the installed runtime

**Purpose:** Verify that the dependency installation completed in the active notebook environment.

**Inputs:** The current Python package environment.

**Outputs:** A visible package/version check.

**Why it matters:** Early feedback prevents later pipeline cells from failing because setup was skipped.

In [0]:
# Purpose: load the central project configuration.

from pathlib import Path
import yaml

project_root = Path(
    "/Workspace/Users/parvinbadalov@softserve.academy/"
    "Databricks-Academy-Lakehouse/Demos/Demo2_Olist"
)

config_path = project_root / "config/project_config.yml"

with config_path.open("r", encoding="utf-8") as config_file:
    project_config = yaml.safe_load(config_file)

catalog = project_config["unity_catalog"]["catalog"]
schema = project_config["unity_catalog"]["schema"]

print("Project root:", project_root)
print("Catalog:", catalog)
print("Schema:", schema)

### Verify the Databricks session

**Purpose:** Inspect the active Spark and Databricks runtime context after restart.

**Inputs:** The current notebook session and installed libraries.

**Outputs:** Runtime diagnostics used to confirm environment readiness.

**Why it matters:** The bundle relies on Databricks-only APIs that should be checked before running transformations.

In [0]:
# Purpose: create the configured schema if it does not already exist.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Environment ready: {catalog}.{schema}")

### Load project configuration

**Purpose:** Read the central YAML configuration for tables, views, governance, and quality settings.

**Inputs:** `config/project_config.yml`.

**Outputs:** Configuration values available to setup and validation logic.

**Why it matters:** Central configuration keeps table contracts and development behavior consistent.

In [0]:
# Purpose: verify that the required Python packages are available.

from importlib.metadata import version

required_packages = [
    "databricks-labs-dqx",
    "pytest",
    "PyYAML",
]

for package_name in required_packages:
    print(f"{package_name}: {version(package_name)}")

print("SUCCESS: Demo2_Olist environment setup completed")

### Create the isolated development schema

**Purpose:** Ensure the configured Unity Catalog schema exists for Demo2 Olist objects.

**Inputs:** Catalog and schema configuration.

**Outputs:** An idempotently available development schema.

**Why it matters:** `IF NOT EXISTS` setup avoids destructive changes while making reruns safe.